In [ ]:
!pip install pyspark

In [ ]:
import csv
import random
from datetime import datetime, timedelta

# --- Configuration ---
NUM_ROWS = 50000  # Adjust this number to make the dataset as large as you need
OUTPUT_FILE = "spark_large_dataset.csv"

# --- Data Pools ---
regions = ["North", "South", "East", "West"]
categories = ["Electronics", "Clothing", "Home", "Sports", "Books"]
statuses = ["Active", "Inactive", "Pending", None] # Includes nulls for Q5
cities = ["New York", "Los Angeles", "San Francisco", "Austin", "Chicago", "Seattle"]
subscriptions = ["Basic", "Premium", "Pro"]

# Helper to generate random dates and timestamps
def random_date(start_year=2023, end_year=2024):
    start = datetime(start_year, 1, 1)
    end = datetime(end_year, 1, 1)
    return start + timedelta(seconds=random.randint(0, int((end - start).total_seconds())))

print(f"Generating {NUM_ROWS} rows of synthetic data...")

with open(OUTPUT_FILE, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)

    # Write the Header
    writer.writerow([
        "user_id", "transaction_date", "region", "product_category", "sale_amount",
        "status", "city", "age", "subscription", "raw_timestamp",
        "email", "username", "price", "store_id"
    ])

    previous_row = None

    for i in range(1, NUM_ROWS + 1):
        # 5% chance to create an exact duplicate row to satisfy Q3 requirements
        if previous_row and random.random() < 0.05:
            writer.writerow(previous_row)
            continue

        # Generate base data
        user_id = f"U{random.randint(1000, 99999)}"
        dt = random_date()
        transaction_date = dt.strftime('%Y-%m-%d')
        raw_timestamp = dt.strftime('%Y-%m-%d %H:%M:%S')

        region = random.choice(regions)
        category = random.choice(categories)
        city = random.choice(cities)
        age = random.randint(16, 65)  # Covers the 18-30 range for Q8
        subscription = random.choice(subscriptions)

        # Sale amount and price (10% chance price is null for Q15)
        sale_amount = round(random.uniform(10.0, 500.0), 2)
        price = sale_amount if random.random() > 0.10 else None

        status = random.choice(statuses)

        # Email (10% chance email is null for Q12)
        email = f"user{random.randint(1, 99999)}@example.com" if random.random() > 0.10 else None

        # Username (10% chance username is an empty string for Q12)
        username = f"user_{random.randint(1, 99999)}" if random.random() > 0.10 else ""

        store_id = f"S{random.randint(1, 50)}"

        # Construct the row
        row = [
            user_id, transaction_date, region, category, sale_amount,
            status, city, age, subscription, raw_timestamp,
            email, username, price, store_id
        ]

        writer.writerow(row)
        previous_row = row

print(f"Dataset successfully saved to your current directory as: {OUTPUT_FILE}")

Generating 50000 rows of synthetic data...
Dataset successfully saved to your current directory as: spark_large_dataset.csv


In [ ]:
from pyspark.sql.functions import col, avg, sum, min, max, mean
from pyspark.sql.types import TimestampType

In [ ]:
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder.appName("SparkTask").getOrCreate()

# Load the dataset generated in the previous steps
df = spark.read.csv("spark_large_dataset.csv", header=True, inferSchema=True)

# Show a preview to confirm it is loaded
df.show(5)

+-------+----------------+------+----------------+-----------+--------+-------------+---+------------+-------------------+--------------------+----------+------+--------+
|user_id|transaction_date|region|product_category|sale_amount|  status|         city|age|subscription|      raw_timestamp|               email|  username| price|store_id|
+-------+----------------+------+----------------+-----------+--------+-------------+---+------------+-------------------+--------------------+----------+------+--------+
| U25321|      2023-06-21|  West|     Electronics|     461.72|Inactive|     New York| 51|     Premium|2023-06-21 09:24:16|user34778@example...|user_60643|461.72|      S2|
| U26631|      2023-04-11| South|        Clothing|     442.17| Pending|     New York| 54|       Basic|2023-04-11 16:19:12|user54026@example...|user_60872|442.17|     S19|
| U62384|      2023-05-12| North|          Sports|     189.83| Pending|       Austin| 26|     Premium|2023-05-12 23:55:37|user36134@example...|us

Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?   

    Disk I/O Bottleneck: Traditional MapReduce writes intermediate processing results to physical disk (HDFS) between every single map and reduce phase.

    Latency: The constant reading and writing to disk creates massive latency. Spark overcomes this by utilizing in-memory processing, making it substantially faster for complex, multi-step data pipelines.

Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.   

    Machine learning algorithms require passing over the same dataset multiple times (iterative processing) to optimize parameters.

    Disk-based systems must reload the data from the disk for every iteration. Spark caches the datasets (DataFrames/RDDs) directly in RAM. Subsequent iterations access this data at memory speed, drastically reducing the overall execution time.

Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

In [ ]:
# Now that 'df' is defined, we can remove duplicates
df_cleaned = df.dropDuplicates(["user_id", "transaction_date"])

print(f"Original count: {df.count()}")
print(f"Cleaned count: {df_cleaned.count()}")
df_cleaned.show(5)

Original count: 50000
Cleaned count: 47438
+-------+----------------+------+----------------+-----------+--------+-------------+---+------------+-------------------+--------------------+----------+------+--------+
|user_id|transaction_date|region|product_category|sale_amount|  status|         city|age|subscription|      raw_timestamp|               email|  username| price|store_id|
+-------+----------------+------+----------------+-----------+--------+-------------+---+------------+-------------------+--------------------+----------+------+--------+
| U10006|      2023-05-22|  East|        Clothing|     454.88| Pending|  Los Angeles| 56|         Pro|2023-05-22 17:36:37|user6939@example.com|user_43560|454.88|     S38|
| U10017|      2023-06-30|  East|     Electronics|     295.88|    NULL|San Francisco| 32|         Pro|2023-06-30 00:13:54|user2102@example.com|user_53240|295.88|     S31|
| U10022|      2023-01-09| North|           Books|     450.86| Pending|       Austin| 47|     Premium|

Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount

In [ ]:
df_west_avg = df.filter(col("region") == "West") \
                      .groupBy("product_category") \
                      .agg(avg("sale_amount").alias("average_sale_amount"))

df_west_avg.show()

+----------------+-------------------+
|product_category|average_sale_amount|
+----------------+-------------------+
|            Home| 253.30747149837106|
|          Sports| 254.40691328560087|
|     Electronics|   254.955607625099|
|        Clothing| 248.99657749469213|
|           Books|  255.1556199677942|
+----------------+-------------------+



Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.   

    .na.drop() removes the entire row if it contains a null value.

    .na.fill() replaces the null value with a specified default value instead of deleting the row.

In [ ]:
df_filled = df.na.fill({"status": "Unknown"})
df_filled.select("user_id", "status").show(5)

+-------+--------+
|user_id|  status|
+-------+--------+
| U25321|Inactive|
| U26631| Pending|
| U62384| Pending|
| U52145|Inactive|
| U52145|Inactive|
+-------+--------+
only showing top 5 rows


Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [ ]:
df_city_counts = df.groupBy("city") \
                   .count() \
                   .filter(col("count") > 100)

df_city_counts.show()

+-------------+-----+
|         city|count|
+-------------+-----+
|  Los Angeles| 8250|
|San Francisco| 8499|
|       Austin| 8405|
|      Chicago| 8259|
|      Seattle| 8253|
|     New York| 8334|
+-------------+-----+



Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?   

    Because DataFrames are immutable, they cannot be modified in place.

    Whenever you execute a cleaning step like .drop() or .withColumnRenamed(), Spark does not alter the original DataFrame; instead, it generates and returns a brand new DataFrame reflecting those changes. You must assign this result to a new variable (or overwrite the existing one) to persist your cleaning steps.

Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [ ]:
df_filtered = df.filter(col("age").between(18, 30) & (col("subscription") == "Premium"))
df_filtered.select("user_id", "age", "subscription").show(5)

+-------+---+------------+
|user_id|age|subscription|
+-------+---+------------+
| U62384| 26|     Premium|
| U25201| 27|     Premium|
| U72367| 22|     Premium|
| U84530| 22|     Premium|
| U64971| 20|     Premium|
+-------+---+------------+
only showing top 5 rows


Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?   

    Null values can skew mathematical results or cause unexpected behaviors. If ignored, aggregate functions might silently drop rows containing nulls, resulting in inaccurate averages based on a smaller denominator. Handling them first (via dropping or filling with defaults) guarantees data integrity.

Q10: Write the code to revise a column named raw_timestamp by casting it to a Timestamp Type and renaming it to event_time.

In [ ]:
df_timestamped = df.withColumn("event_time", col("raw_timestamp").cast(TimestampType())) \
                   .drop("raw_timestamp")

df_timestamped.select("user_id", "event_time").show(5)

+-------+-------------------+
|user_id|         event_time|
+-------+-------------------+
| U25321|2023-06-21 09:24:16|
| U26631|2023-04-11 16:19:12|
| U62384|2023-05-12 23:55:37|
| U52145|2023-03-02 16:15:38|
| U52145|2023-03-02 16:15:38|
+-------+-------------------+
only showing top 5 rows


Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?   

    The Shuffle: A shuffle redistributes data across different partitions (and often different physical worker nodes) to ensure all records sharing the same key are grouped together for aggregation.

    Wide Transformation: It is considered a "wide" transformation because computing a single output partition requires reading and moving data from multiple input partitions over the network, making it a highly resource-intensive operation.

Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [ ]:
df_valid_users = df.filter(col("email").isNotNull() & (col("username") != ""))
print(f"Rows with valid emails/usernames: {df_valid_users.count()}")
df_valid_users.select("user_id", "email", "username").show(5)

Rows with valid emails/usernames: 40600
+-------+--------------------+----------+
|user_id|               email|  username|
+-------+--------------------+----------+
| U25321|user34778@example...|user_60643|
| U26631|user54026@example...|user_60872|
| U62384|user36134@example...|user_66887|
| U52145|user68346@example...|user_15473|
| U52145|user68346@example...|user_15473|
+-------+--------------------+----------+
only showing top 5 rows


Q13: How do you use the agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

In [ ]:
df_stats = df.agg(
    min("price").alias("min_price"),
    max("price").alias("max_price"),
    mean("price").alias("mean_price")
)

df_stats.show()

+---------+---------+------------------+
|min_price|max_price|        mean_price|
+---------+---------+------------------+
|    10.01|    500.0|255.21345597547227|
+---------+---------+------------------+



Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?   

    When inferSchema=true encounters inconsistent date strings (e.g., mixing MM/DD/YYYY and YYYY-MM-DD), it will fail to recognize them as standard dates.

    As a fallback, Spark will ingest the column as a generic StringType. This forces you to write complex, custom parsing logic later to cast them properly, rather than handling them as native Date objects from the start.

Q15: Write a final processing pipeline that: 1. Filters out duplicates. 2. Fills null prices with 0. 3. Groups by store_id to calculate total revenue.

In [ ]:
final_pipeline_df = df.dropDuplicates() \
                      .na.fill({"price": 0.0}) \
                      .groupBy("store_id") \
                      .agg(sum("price").alias("total_revenue"))

final_pipeline_df.orderBy("total_revenue", ascending=False).show(5)

+--------+------------------+
|store_id|     total_revenue|
+--------+------------------+
|     S13|239669.81000000003|
|     S29|238157.82000000012|
|     S17|230727.66000000006|
|     S45|230329.77999999997|
|     S24|230104.23000000007|
+--------+------------------+
only showing top 5 rows
